
# 07 Chance-Adjusted Reporting and Visualisation

This notebook generates publication-style figures and tables for the
chance-adjusted pseudo-concept evaluation branch.

Inputs:
- outputs/pilot_1260/metrics/chance_adjusted/*.csv

Outputs:
- outputs/phase5_pilot_1260/figures/chance_adjusted/
- outputs/phase5_pilot_1260/tables/chance_adjusted/

Main goals:
1. Compare raw overlap vs chance-adjusted enrichment
2. Visualise sparse pseudo-concept behaviour
3. Investigate lesion-size sensitivity
4. Compare melanoma vs non-melanoma enrichment
5. Visualise random baseline distributions


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()

OUTPUTS_DIR = ROOT / "outputs"
RUN_NAME = "pilot_1260_strat"
PHASE5_DIR = OUTPUTS_DIR / f"phase5_{RUN_NAME}"

METRICS_DIR = PHASE5_DIR / "metrics"
CHANCE_DIR = METRICS_DIR / "chance_adjusted"

FIG_DIR = PHASE5_DIR / "figures" / "reporting"
TABLE_DIR = PHASE5_DIR / "tables" / "reporting"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Metrics dir:", METRICS_DIR)
print("Figures dir:", FIG_DIR)
print("Tables dir:", TABLE_DIR)


In [ ]:

chance_metrics = pd.read_csv(
    CHANCE_DIR / "phase5_pilot_1260_chance_adjusted_xai_concept_metrics.csv"
)

summary_method_concept = pd.read_csv(
    CHANCE_DIR / "phase5_pilot_1260_chance_summary_by_method_concept.csv"
)

summary_size = pd.read_csv(
    CHANCE_DIR / "phase5_pilot_1260_chance_summary_by_size_class.csv"
)

summary_label = pd.read_csv(
    CHANCE_DIR / "phase5_pilot_1260_chance_summary_by_label.csv"
)

random_dist = pd.read_csv(
    CHANCE_DIR / "phase5_pilot_1260_chance_random_distributions.csv"
)

print(chance_metrics.shape)
print(summary_method_concept.shape)
print(summary_size.shape)
print(summary_label.shape)
print(random_dist.shape)



# 1. Summary Table by Method and Concept


In [ ]:

summary_method_concept.head()

summary_method_concept.to_csv(
    TABLE_DIR / "chance_summary_by_method_concept.csv",
    index=False
)

summary_method_concept.head(20)
print(summary_method_concept.columns)
print(summary_size.columns)





# 2. Histogram: Enrichment by Method and Concept


In [ ]:

# Cleaner plot: chance-adjusted enrichment by concept and method


metric_col = ["mean_dice_enrichment", "mean_iou_enrichment", "mean_sir_enrichment", "mean_tixai_enrichment"]
for col in metric_col:

    selected_concepts = [
        "asymmetry",
        "border_default",
        "colour_heterogeneity",
    ]

    plot_df = summary_method_concept[
        summary_method_concept["concept"].isin(selected_concepts)
    ].copy()

    plot_df = (
        plot_df
        .groupby(["concept", "xai_method"], as_index=False)[col]
        .mean()
    )

    plot_df[col] = plot_df[col].clip(lower=-2, upper=5)

    fig, ax = plt.subplots(figsize=(9, 5))

    concepts = selected_concepts
    methods = sorted(plot_df["xai_method"].unique())

    x = np.arange(len(concepts))
    width = 0.25

    for i, method in enumerate(methods):
        vals = []

        for concept in concepts:
            sub = plot_df[
                (plot_df["concept"] == concept) &
                (plot_df["xai_method"] == method)
            ]
            vals.append(sub[col].mean() if len(sub) else np.nan)

        ax.bar(
            x + i * width,
            vals,
            width,
            label=method
        )

    ax.axhline(1.0, linestyle="--", linewidth=1)

    ax.set_xticks(x + width)
    ax.set_xticklabels(
        ["Asymmetry", "Border", "Colour heterogeneity"],
        rotation=20,
        ha="right"
    )

    ax.set_ylabel(col)
    ax.set_title("Chance-adjusted pseudo-concept enrichment")
    ax.legend(title="XAI method")

    plt.tight_layout()

    out_path = FIG_DIR / "bar_chance_adjusted_enrichment_selected_concepts.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.show()

    print("Saved:", out_path)


# 3. Lesion Size Sensitivity

Visualise how enrichment changes across lesion size classes.


In [ ]:

# Lesion-size sensitivity

metric_cols = [
    "mean_dice_enrichment",
    "mean_iou_enrichment",
    "mean_sir_enrichment",
]

concept_to_plot = "asymmetry"

size_order = [
    "very_tiny",
    "tiny",
    "small",
    "normal",
]

for col in metric_cols:

    plot_df = summary_size[
        summary_size["concept"] == concept_to_plot
    ].copy()

    plot_df["mask_size_class"] = pd.Categorical(
        plot_df["mask_size_class"],
        categories=size_order,
        ordered=True
    )

    plot_df = plot_df.sort_values("mask_size_class")

    fig, ax = plt.subplots(figsize=(9, 5))

    methods = sorted(plot_df["xai_method"].unique())

    x = np.arange(len(size_order))
    width = 0.8 / len(methods)

    for i, method in enumerate(methods):

        sub = plot_df[
            plot_df["xai_method"] == method
        ]

        vals = []

        for size in size_order:

            row = sub[
                sub["mask_size_class"] == size
            ]

            vals.append(
                row[col].mean() if len(row) else np.nan
            )

        ax.bar(
            x + i * width,
            vals,
            width,
            label=method
        )

    ax.set_xticks(
        x + width * (len(methods) - 1) / 2
    )

    ax.set_xticklabels(size_order)

    ax.set_title(
        f"{concept_to_plot} - {col} across lesion sizes"
    )

    ax.set_xlabel("Lesion size class")
    ax.set_ylabel(col)

    ax.legend()

    plt.tight_layout()

    out_path = FIG_DIR / f"{concept_to_plot}_{col}_size_sensitivity.png"

    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", out_path)


# 4. Melanoma vs Non-Melanoma Comparison


In [ ]:
metric_cols = [
    "mean_dice_enrichment",
    "mean_iou_enrichment",
    "mean_sir_enrichment",
]


label_metric_col = metric_col

concept_to_plot = "colour_heterogeneity"
for label_metric_col in metric_cols:

    plot_df = summary_label[
        summary_label["concept"] == concept_to_plot
    ].copy()

    fig, ax = plt.subplots(figsize=(10, 6))

    methods = sorted(plot_df["xai_method"].unique())
    labels = sorted(plot_df["label_name"].unique())

    x = np.arange(len(methods))
    width = 0.35

    for idx, label in enumerate(labels):
        vals = []

        for m in methods:
            sub = plot_df[
                (plot_df["xai_method"] == m) &
                (plot_df["label_name"] == label)
            ]

            vals.append(sub[label_metric_col].mean())

        ax.bar(
            x + idx * width,
            vals,
            width=width,
            label=label
        )

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(methods)

    ax.set_ylabel(label_metric_col)

    ax.set_title(
        f"{concept_to_plot} - melanoma vs non-melanoma"
    )

    ax.legend()

    plt.tight_layout()

    out_path = FIG_DIR / "label_comparison_colour_heterogeneity.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.show()

    print("Saved:", out_path)



# 5. Random Baseline Distribution

Histogram of random overlap distribution with observed overlap shown as a vertical line.


In [ ]:

candidate_random_cols = [
    "random_dice",
    "dice_random",
    "random_metric",
]

random_col = None

for c in candidate_random_cols:
    if c in random_dist.columns:
        random_col = c
        break

print("Random column:", random_col)

if random_col is not None:

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.hist(random_dist[random_col].dropna(), bins=30)

    observed_col = None

    for c in ["observed_dice", "dice", "observed_metric"]:
        if c in chance_metrics.columns:
            observed_col = c
            break

    if observed_col is not None:
        obs_val = chance_metrics[observed_col].mean()

        ax.axvline(
            obs_val,
            linestyle="--",
            linewidth=2
        )

    ax.set_title("Random baseline distribution")

    plt.tight_layout()

    out_path = FIG_DIR / "random_distribution_histogram.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.show()

    print("Saved:", out_path)

else:
    print("No recognised random distribution column found.")



# 6. Raw vs Chance-Adjusted Comparison


In [ ]:
# ============================================================
# Raw overlap vs chance-adjusted enrichment
# ============================================================

raw_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_method_concept.csv"

if not raw_summary_path.exists():
    raise FileNotFoundError(f"Missing file: {raw_summary_path}")

raw_summary = pd.read_csv(raw_summary_path)

print("Raw columns:")
print(raw_summary.columns.tolist())

print("\nChance-adjusted columns:")
print(summary_method_concept.columns.tolist())

# ------------------------------------------------------------
# Select comparable raw and chance-adjusted metrics
# ------------------------------------------------------------

raw_col = None
for c in [
    "mean_dice",
    "dice_mean",
    "dice",
    "mean_observed_dice",
]:
    if c in raw_summary.columns:
        raw_col = c
        break

chance_col = None
for c in [
    "mean_dice_enrichment",
    "dice_enrichment",
]:
    if c in summary_method_concept.columns:
        chance_col = c
        break

if raw_col is None:
    raise ValueError(
        "Could not find a Dice column in raw_summary. "
        "Check raw_summary.columns."
    )

if chance_col is None:
    raise ValueError(
        "Could not find a Dice enrichment column in summary_method_concept. "
        "Check summary_method_concept.columns."
    )

print("Using raw column:", raw_col)
print("Using chance-adjusted column:", chance_col)

# ------------------------------------------------------------
# Aggregate by method
# ------------------------------------------------------------

raw_plot = (
    raw_summary
    .groupby("xai_method")[raw_col]
    .mean()
    .rename("mean_raw_dice")
)

chance_plot = (
    summary_method_concept
    .groupby("xai_method")[chance_col]
    .mean()
    .rename("mean_dice_enrichment")
)

compare_df = pd.concat(
    [raw_plot, chance_plot],
    axis=1
)

display(compare_df)

# ------------------------------------------------------------
# Plot with two y-axes because scales differ
# ------------------------------------------------------------

fig, ax1 = plt.subplots(figsize=(8, 5))

x = np.arange(len(compare_df.index))
width = 0.35

ax1.bar(
    x - width / 2,
    compare_df["mean_raw_dice"],
    width,
    label="Mean raw Dice"
)

ax1.set_ylabel("Mean raw Dice")
ax1.set_xticks(x)
ax1.set_xticklabels(compare_df.index)
ax1.set_ylim(0, max(compare_df["mean_raw_dice"].max() * 1.3, 0.05))

ax2 = ax1.twinx()

ax2.bar(
    x + width / 2,
    compare_df["mean_dice_enrichment"],
    width,
    label="Mean Dice enrichment"
)

ax2.axhline(1.0, linestyle="--", linewidth=1)

ax2.set_ylabel("Mean Dice enrichment")

fig.suptitle("Raw Dice vs Chance-Adjusted Dice Enrichment")

# combined legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper right"
)

plt.tight_layout()

out_path = FIG_DIR / "raw_dice_vs_chance_adjusted_dice_enrichment.png"

plt.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", out_path)

In [ ]:
raw_col = "mean_dice"              # check this exists in raw_summary
chance_col = "mean_dice_enrichment"  # exists in summary_method_concept

raw_plot = (
    raw_summary
    .groupby("xai_method")[raw_col]
    .mean()
    .rename("raw_overlap")
)

chance_plot = (
    summary_method_concept
    .groupby("xai_method")[chance_col]
    .mean()
    .rename("chance_adjusted")
)

compare_df = pd.concat([raw_plot, chance_plot], axis=1)

display(compare_df)

In [ ]:
compare_df.plot(kind="bar", figsize=(8, 6))

plt.axhline(1.0, linestyle="--", linewidth=1)
plt.title("Raw Dice vs Chance-Adjusted Dice Enrichment")
plt.ylabel("Metric value")

plt.tight_layout()

out_path = FIG_DIR / "raw_vs_chance_adjusted.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")

plt.show()
print("Saved:", out_path)

In [ ]:
# ============================================================
# Raw Dice vs Chance-Adjusted Dice Enrichment
# Dual y-axis version
# ============================================================

fig, ax1 = plt.subplots(figsize=(8, 5))

x = np.arange(len(compare_df.index))
width = 0.35

# ------------------------------------------------------------
# Left axis: raw Dice
# ------------------------------------------------------------

bars1 = ax1.bar(
    x - width / 2,
    compare_df["raw_overlap"],
    width,
    label="Raw Dice overlap",
    color="tab:blue"
)

ax1.set_ylabel("Mean raw Dice")
ax1.set_ylim(0, 0.06)

ax1.set_xticks(x)
ax1.set_xticklabels(compare_df.index)

# ------------------------------------------------------------
# Right axis: enrichment
# ------------------------------------------------------------

ax2 = ax1.twinx()

bars2 = ax2.bar(
    x + width / 2,
    compare_df["chance_adjusted"],
    width,
    label="Dice enrichment",
    color="tab:orange"
)

ax2.set_ylabel("Mean Dice enrichment")
ax2.set_ylim(0, 1.2)

# Chance-level reference
ax2.axhline(
    1.0,
    linestyle="--",
    linewidth=1
)

# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

plt.title(
    "Raw Dice overlap vs chance-adjusted Dice enrichment"
)

# ------------------------------------------------------------
# Combined legend
# ------------------------------------------------------------

handles = [bars1, bars2]
labels = ["Raw Dice overlap", "Dice enrichment"]

ax1.legend(handles, labels, loc="upper right")

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

out_path = FIG_DIR / "raw_vs_chance_adjusted_dual_axis.png"

plt.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", out_path)


# 7. Export Core Summary Tables


In [ ]:

summary_method_concept.to_csv(
    TABLE_DIR / "summary_method_concept.csv",
    index=False
)

summary_size.to_csv(
    TABLE_DIR / "summary_size_class.csv",
    index=False
)

summary_label.to_csv(
    TABLE_DIR / "summary_label.csv",
    index=False
)

print("Exported reporting tables.")
